# 1. Creating table

In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.dim_geonational AS
WITH distinct_geo AS (
    SELECT DISTINCT
        geographic_area
    FROM cpt_utility_catalog.silver.silver_cpi_cleaned
    WHERE geographic_area IS NOT NULL
),
geo_final AS(
    SELECT
    xxhash64(LOWER(TRIM(geographic_area))) AS geography_key,
    geographic_area AS region_name, -- region name
    
    -- region code
    CASE 
        WHEN geographic_area LIKE '%Total Country%' THEN 'ZA'
        WHEN geographic_area LIKE '%Urban Areas%'   THEN 'URB'
        WHEN geographic_area LIKE '%Rural Areas%'   THEN 'RUR'
        WHEN geographic_area LIKE '%Eastern Cape%'  THEN 'EC'
        WHEN geographic_area LIKE '%Free State%'    THEN 'FS'
        WHEN geographic_area LIKE '%Gauteng%'       THEN 'GP'
        WHEN geographic_area LIKE '%Kwazulu-Natal%' THEN 'KZN'
        WHEN geographic_area LIKE '%Limpopo%'       THEN 'LP'
        WHEN geographic_area LIKE '%Mpumalanga%'    THEN 'MP'
        WHEN geographic_area LIKE '%Northern Cape%' THEN 'NC'
        WHEN geographic_area LIKE '%North-West%'    THEN 'NW'
        WHEN geographic_area LIKE '%Western Cape%'  THEN 'WC'
        ELSE 'OTHER'
    END AS region_code,

    -- region type
    CASE 
        WHEN geographic_area LIKE '%Total Country%' THEN 'Country'
        WHEN geographic_area LIKE '%Urban%' 
          OR geographic_area LIKE '%Rural%'         THEN 'Settlement Aggregate'
        ELSE 'Province'
    END AS region_type,

    TRUE AS is_active
    FROM distinct_geo
)
SELECT * FROM geo_final

UNION ALL

SELECT
    xxhash64('unmapped') AS geography_key,
    'Unmapped' AS region_name,
    'Unmapped' AS region_code,
    'Unmapped' AS region_type,
    TRUE AS is_active